# Before Week 1 — test OpenRouter and Ollama

Run each cell in order. The aim is to see two successful model calls, not to understand every setup line yet. Never display the OpenRouter key. Only synthetic text is sent.

The two routes perform the same broad operation but in different places: OpenRouter sends the request to a hosted provider; Ollama sends it to a model server on this computer.

OpenRouter and Ollama are services, not model names. The `openrouter` and `ollama` packages are Python clients. They translate ordinary Python objects into API requests and translate the replies back into Python response objects.

## What this notebook is checking

The first cells check one layer at a time: Python version → course configuration → installed SDKs → hidden key → local server → downloaded model. This matters because an error at one layer does not mean every other layer is broken.

**Hosted path:** notebook → `openrouter` client → internet → OpenRouter → model provider → response object.

**Local path:** notebook → `ollama` client → `localhost:11434` → downloaded model → response object.

In [ ]:
import json
import os
import sys
from getpass import getpass
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Supported version:', (3, 11) <= sys.version_info[:2] < (3, 13))

In [ ]:
repo_root = Path.cwd()
while not (repo_root / 'config' / 'course_models.json').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

config = json.loads((repo_root / 'config' / 'course_models.json').read_text())
HOSTED_MODEL = config['hosted']['model']
LOCAL_MODEL = config['local']['model']
print('Hosted model:', HOSTED_MODEL)
print('Local model:', LOCAL_MODEL)

In [ ]:
from openrouter import OpenRouter
import ollama
print('✅ Both Python SDKs imported.')

In [ ]:
if not os.getenv('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter course key (hidden): ')
print('✅ OpenRouter key is available:', bool(os.getenv('OPENROUTER_API_KEY')))

In [ ]:
local_models = [item.model for item in ollama.list().models]
print('Ollama is reachable:', True)
print('Course model installed:', LOCAL_MODEL in local_models)
print('If False, run: ollama pull ' + LOCAL_MODEL)

## Make the two test calls

**Input:** one synthetic instruction stored in `test_message`. **Output:** one short text response from each route. The message list is an ordinary Python list containing a dictionary with `role` and `content` fields.

### OpenRouter: what the next cell connects to

`OpenRouter(...)` creates a client and supplies the hidden credential. `client.chat.send(...)` sends the model string, message list and generation settings to a remote API. OpenRouter then routes the request to a provider serving that model. The return is an SDK object; `choices[0].message.content` retrieves its first returned text.

In [ ]:
test_message = 'Reply with exactly: ROUTE READY'
hosted_messages = [{'role': 'user', 'content': test_message}]

with OpenRouter(api_key=os.environ['OPENROUTER_API_KEY']) as client:
    hosted_response = client.chat.send(
        model=HOSTED_MODEL,
        messages=hosted_messages,
        temperature=0,
        max_tokens=12,
    )

hosted_answer = hosted_response.choices[0].message.content.strip()
print('✅ OpenRouter reachable:', bool(hosted_answer))
print('Hosted output:', hosted_answer)

### Ollama: what the next cell connects to

`ollama.chat(...)` sends the model string, same broad message structure and local options to the Ollama API at `localhost`. Ollama finds the downloaded model and performs inference on this computer. No OpenRouter key is sent. The return has a different shape, so its text is retrieved through `local_response.message.content`.

In [ ]:
local_messages = [{'role': 'user', 'content': test_message}]
local_response = ollama.chat(
    model=LOCAL_MODEL,
    messages=local_messages,
    options={'temperature': 0},
)
local_answer = local_response.message.content.strip()
print('✅ Ollama reachable:', bool(local_answer))
print('Local output:', local_answer)

## Setup completion evidence

Take one screenshot containing the two green reachable lines. Do not include the key-entry cell, terminal history, usernames or machine details. If the local model cannot run on your machine, bring the error to class and complete the Ollama call on the paired or instructor machine.